# sql-forge — Dataset Exploration

Explore the generated SFT and DPO datasets before training.

In [ ]:
import json
import sys
from pathlib import Path
import re
from collections import Counter

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data' / 'dataset'
SCHEMA_DIR = ROOT / 'data' / 'schemas'

print('ROOT:', ROOT)
print('DATA_DIR exists:', DATA_DIR.exists())

## 1. Load datasets

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

sft_train = load_jsonl(DATA_DIR / 'sft_train.jsonl')
sft_test  = load_jsonl(DATA_DIR / 'sft_test.jsonl')
dpo_pairs = load_jsonl(DATA_DIR / 'dpo_pairs.jsonl')

print(f'SFT train: {len(sft_train)} examples')
print(f'SFT test:  {len(sft_test)} examples')
print(f'DPO pairs: {len(dpo_pairs)} pairs')

## 2. SFT dataset — sample examples

In [ ]:
import random
sample = random.choice(sft_train)
print('=== PROMPT ===')
print(sample['prompt'])
print('\n=== COMPLETION ===')
print(sample['completion'])

## 3. Token length distribution

In [ ]:
def approx_tokens(text):
    """Rough token count: split on whitespace + punctuation."""
    return len(re.findall(r'\S+', text))

sft_lengths = [approx_tokens(ex['text']) for ex in sft_train]

print(f'SFT text lengths (approx tokens):')
print(f'  min:    {min(sft_lengths)}')
print(f'  max:    {max(sft_lengths)}')
print(f'  mean:   {sum(sft_lengths)/len(sft_lengths):.0f}')
print(f'  >1024:  {sum(1 for l in sft_lengths if l > 1024)} ({sum(1 for l in sft_lengths if l > 1024)/len(sft_lengths)*100:.1f}%)')
print(f'  >2048:  {sum(1 for l in sft_lengths if l > 2048)} ({sum(1 for l in sft_lengths if l > 2048)/len(sft_lengths)*100:.1f}%)')

## 4. BigQuery keyword coverage

In [ ]:
BQ_KEYWORDS = [
    'QUALIFY', 'UNNEST', 'ARRAY_AGG', 'STRUCT', 'COUNTIF',
    'FORMAT_DATE', 'DATE_TRUNC', 'TIMESTAMP_SUB', 'TIMESTAMP_DIFF',
    'PERCENTILE_CONT', 'SAFE_DIVIDE', 'JSON_VALUE', 'ARRAY_LENGTH',
    'TIMESTAMP_TRUNC', 'TIMESTAMP_ADD', 'DATE_DIFF', 'DATE_SUB',
    'EXTRACT', 'LAST_DAY', 'NULLIF', 'COALESCE',
    'ROW_NUMBER', 'RANK', 'DENSE_RANK', 'NTILE', 'PERCENT_RANK',
    'LAG', 'LEAD', 'WITH',
]

all_sql = ' '.join(ex['completion'].upper() for ex in sft_train)
kw_counts = {kw: all_sql.count(kw) for kw in BQ_KEYWORDS}

print('BigQuery keyword frequency in SFT train set:')
for kw, count in sorted(kw_counts.items(), key=lambda x: -x[1]):
    bar = '#' * min(count // 5, 40)
    print(f'  {kw:<22} {count:>5}  {bar}')

## 5. DPO rejection type distribution

In [ ]:
rejection_types = Counter(p.get('rejection_type', 'unknown') for p in dpo_pairs)
print('DPO rejection type distribution:')
for rtype, count in rejection_types.most_common():
    pct = count / len(dpo_pairs) * 100
    bar = '#' * int(pct / 2)
    print(f'  {rtype:<25} {count:>4} ({pct:>5.1f}%)  {bar}')

## 6. Sample DPO pair — chosen vs rejected

In [ ]:
pair = random.choice(dpo_pairs)
print('=== PROMPT ===')
print(pair['prompt'])
print('\n=== CHOSEN (correct BigQuery) ===')
print(pair['chosen'])
print('\n=== REJECTED (type:', pair.get('rejection_type', '?'), ') ===')
print(pair['rejected'])

## 7. SQL parse validity check

In [ ]:
import sqlglot

DBT_JINJA_RE = re.compile(r'\{\{.*?\}\}|\{%.*?%\}', re.DOTALL)

def is_parseable(sql: str) -> bool:
    clean = DBT_JINJA_RE.sub('1', sql)
    try:
        result = sqlglot.parse(clean, dialect='bigquery')
        return len(result) > 0 and result[0] is not None
    except Exception:
        return False

train_sqls = [ex['completion'].strip() for ex in sft_train]
parse_results = [is_parseable(sql) for sql in train_sqls]
parse_rate = sum(parse_results) / len(parse_results)

print(f'SFT train parse accuracy (sqlglot BigQuery): {parse_rate:.1%}')
print(f'  Parseable:   {sum(parse_results)}')
print(f'  Unparseable: {len(parse_results) - sum(parse_results)}')

## 8. Gold set overview

In [ ]:
gold_set = load_jsonl(ROOT / 'eval' / 'gold_set.jsonl')
print(f'Gold set: {len(gold_set)} held-out examples')

gold_keywords = Counter()
for ex in gold_set:
    sql_upper = ex['sql'].upper()
    for kw in BQ_KEYWORDS:
        if kw in sql_upper:
            gold_keywords[kw] += 1

print('\nBigQuery patterns covered in gold set:')
for kw, count in gold_keywords.most_common():
    print(f'  {kw:<22} {count:>3} examples')

## 9. Schema file inventory

In [ ]:
import yaml

for schema_file in sorted(SCHEMA_DIR.glob('*.yaml')):
    with open(schema_file) as f:
        schema = yaml.safe_load(f)
    col_types = Counter(col['type'].split('<')[0] for col in schema.get('columns', []))
    print(f"{schema['table']:<25} {len(schema.get('columns', []))} cols  "
          f"{dict(col_types)}")